# 2.6 同步指令：让积木严丝合缝

六块积木的最后一块。指令是异步发射的（2.1 节），搬运与计算分属不同执行单元——**谁来保证“货到了才开工”**？答案是同步机制。理解它，你就理解了 Ascend C 队列 API 的全部“魔法”。

## 1. 为什么必须有同步

回看 2.1 节的三条流：Scalar 把指令发射到各单元的**独立队列**后就撒手不管了。于是存在这样的竞态：

```text
t0: MTE2 收到「搬 Q 块到 L1」        （搬运耗时 100 拍）
t1: MTE1 收到「L1 → L0B 分形搬运」    ← 此时 Q 块可能还没到齐！
t2: Cube  收到「mmad」                ← 此时数据可能是垃圾
```

同步指令的作用：在指令流中插入**依赖标记**，强制后序指令等待前序完成。

## 2. PIPE 与事件：SetFlag / WaitFlag

每类执行单元对应一条 **PIPE**（指令流水线）：

| PIPE | 执行单元 | FA 中的典型指令 |
|--|--|--|
| PIPE_MTE2 | 外部搬运单元 | Q/K/V 块 GM → L1 |
| PIPE_MTE1 | 内部搬运单元 | L1 → L0A/B 分形搬运 |
| PIPE_V | Vector Core | Exp / ReduceMax / Mul… |
| PIPE_MTE3 | 输出搬运单元 | 结果 UB → GM |
| PIPE_M / PIPE_FIX | MTE3/FIXPIPE 的内部级 | FIXPIPE 随路量化 |
| PIPE_S | Scalar | 循环与分支 |

同步原语是一对“暗号”：**SetFlag**（生产者插旗：“我做完了”）与 **WaitFlag**（消费者等待：“旗没插好我不动”）。每对 PIPE 间有独立的**事件 ID**，可同时挂多组依赖互不干扰：

```text
PIPE_MTE2:  CopyIn(qBlock)  ──SetFlag(E0)──▶
PIPE_MTE1:                 WaitFlag(E0)──▶ Split(qBlock) ──SetFlag(E1)──▶
PIPE_V/Cube:                              WaitFlag(E1)──▶ Compute ──SetFlag(E2)──▶
PIPE_MTE3:                                             WaitFlag(E2)──▶ CopyOut
```

> **FA Kernel 的每一次迭代**都是这条链的一次穿越。手写 SetFlag/WaitFlag 繁琐易错——于是有了封装。

## 3. TQue 队列：同步的工程化封装

Ascend C 用 **TPipe + TQue** 把同步细节包起来：

<img src="./images/tque_queue.png" width="600">

> **看图要点**：TQue 是一条“先进先出”的任务队列。生产者（如 MTE2 搬运）完成后 `EnQue`；消费者（如 Vector 计算）先 `DeQue` 再计算。**队列的底层就是 SetFlag/WaitFlag 的事件链**——每个 `EnQue` 隐含一次 SetFlag，每个 `DeQue` 隐含一次 WaitFlag。程序员只需对队列编程，无需手插事件。

```cpp
TPipe pipe;
pipe.InitBuffer(inQueueQ, 2, tileSize * sizeof(half));   // 2 = Double Buffer
// 循环体
LocalTensor<half> q = inQueueQ.DeQue<half>();  // 内含 WaitFlag：等 MTE2 完成
Exp(q, q, cnt, 1);                              // Vector 计算
outQueue.EnQue<half>(q);                        // 内含 SetFlag：通知 MTE3 可搬出
```

## 4. 底层时序：Alloc/EnQue 的真面目

想看穿队列内部，只需两张时序图：

<img src="./images/sync_alloc_timeline.png" width="600">

> **看图要点（AllocTensor）**：`AllocTensor` 内部 = **WaitFlag（等消费者释放旧缓冲）**。双缓冲下，若两份缓冲都被占用，第三次分配必须等最早的消费者搬完——**“内存复用同步”**。

<img src="./images/sync_enque_timeline.png" width="600">

> **看图要点（EnQue/FreeTensor）**：`EnQue` 隐含 **SetFlag**，`FreeTensor` 释放缓冲并唤醒等待的 AllocTensor——**“阶段间时序同步”**。两类同步问题（时序 + 内存复用）在队列 API 中都被优雅解决。第7~8章调优时若怀疑流水气泡，就回到这两张图分析等待链。

## 5. 950 的 C API 同步

950 的纯 C API 保留了同步原语的“裸形态”（也是理解底层的捷径）：

```cpp
asc_copy_gm2ub_align(dst, src, size);           // MTE2 搬入
asc_sync_notify(PIPE_MTE2, PIPE_V, EVENT_ID0);  // 生产者插旗
asc_sync_wait(PIPE_MTE2, PIPE_V, EVENT_ID0);    // 消费者等旗
/* ... Vector 计算 ... */
asc_sync_notify(PIPE_V, PIPE_MTE3, EVENT_ID0);
asc_copy_ub2gm_align(dst, src, size);           // MTE3 搬出
asc_sync();                                      // Kernel 末尾全局同步
```

> 记住这条等式：**TQue 的 EnQue/DeQue ≙ asc_sync_notify/wait 的成对使用**。高阶 API（TQue）与 C API（裸同步）是同一硬件机制的两种语言。

## 章末收官：六块积木，拼出一切

| 积木 | 小节 | 一句话 |
|--|--|--|
| 数据流 | 2.1 | 三条流 + 搬运-计算-搬运 |
| 内存 | 2.2 | 四级存储，私有领地，Tiling 即预算 |
| 搬运 | 2.3 | MTE 四兄弟 + NDDMA，双缓冲开流水 |
| Cube | 2.4 | mmad + M/N/K，五段式范式 |
| Vector | 2.5 | Repeat/Mask/Stride，Softmax 引擎 |
| 同步 | 2.6 | SetFlag/WaitFlag，TQue 即队列化同步 |

**你现在具备了读懂 FA Kernel 的一切基础词汇。** 接下来：
- 若已学过第2章 → 直接做 [1.9 章节测试](01.09_chapter_test.ipynb) 检验本章；
- 若尚未学过 → 进入[第2章 FlashAttention 原理](../02_fa_principle/README.md)。

## 课后练习

1.（单选）`DeQue` 的底层等价于？（A. SetFlag + 通知 MTE3　B. WaitFlag：等待生产者完成　C. mmad 指令　D. InitBuffer）

2.（判断）双缓冲开启后，第三次 AllocTensor 若两份缓冲均被占用，必须等待最早的消费者释放。

3.（单选）FA Kernel 一次迭代中，“QK^T 结果就绪后 Vector 才能 Softmax”这一依赖，用队列 API 表达即？（A. inQueue.EnQue(S) 后 outQueue.DeQue　B. asc_sync()　C. SetFlag 至 PIPE_S　D. 无法表达）

> 完成后查看 [answer/01.08_answer.txt](answer/01.08_answer.txt) 核对答案。